# 🌾 Taller de Clasificación de Cultivos — en tu navegador

Clasifica parcelas agrícolas reales del **Valle del Yaqui (Sonora, México)**
con datos satelitales abiertos — cada paso corre **dentro de tu navegador**
(WebAssembly), sin instalar nada y sin cuentas.

> **Prefer English?** Open
> [`Crop_Classification_Workshop.ipynb`](Crop_Classification_Workshop.ipynb) —
> it is the same workshop, in English.

Este taller es el acompañante didáctico de
[**geocrop_analysis_mx**](https://github.com/abxda/geocrop_analysis_mx),
un pipeline de producción para clasificación de cultivos con datos abiertos
STAC/COG. Lo que aquí corres sobre un tile pequeño, ese pipeline lo corre
sobre regiones completas.

**Autor:** Dr. Abel Coronado ([@abxda](https://github.com/abxda)) ·
construido con Claude Fable (Anthropic)

![dónde corre esto](anim/es/01_where_it_runs.svg)


In [ ]:
# Paso 0 — ¿Dónde está corriendo este Python?
import sys, platform
print(f"Python     : {sys.version.split()[0]}")
print(f"Plataforma : {sys.platform!r} / {platform.machine()!r}")
if sys.platform == "emscripten":
    print("Corriendo en WebAssembly, DENTRO de tu navegador. Sin servidor. 🚀")
else:
    print("Corriendo en modo local (Python normal) — todo funciona igual.")

## ¿De dónde salen los datos?

El tile que vas a cargar es un **producto real**: la **geomediana de
marzo-2018** del Valle del Yaqui, construida por `geocrop_analysis_mx` con
imágenes **HLS de la NASA** (Landsat + Sentinel-2 armonizados), transmitidas
desde catálogos abiertos **STAC/COG**. No se necesitó Google Earth Engine —
aunque el pipeline puede usar GEE opcionalmente si tienes cuenta.

![fuentes de datos](anim/es/02_data_sources.svg)

![la geomediana](anim/es/03_geomedian.svg)


In [ ]:
# Paso 1 — Herramientas + datos del taller (pocos MB; queda en caché)
%pip install -q shepherd-wasm
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"{RAW}/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

RAW = "https://raw.githubusercontent.com/abxda/portable-geocrop/main/files"
TILE     = await trae_archivo("crop_tile_384.tif")
LABELS   = await trae_archivo("crop_labels_384.tif")
NOMBRES  = await trae_archivo("class_names.json")
print("Listo:", TILE, LABELS, NOMBRES)

## Mira el campo: color verdadero y NDVI

13 capas por píxel: 6 bandas espectrales más 7 índices de vegetación/suelo,
ya calculados en la geomediana.

![bandas y NDVI](anim/es/04_bands_ndvi.svg)


In [ ]:
# Paso 2 — RGB y NDVI
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()                      # (13, 384, 384) int16
    nombres_banda = list(src.descriptions)
print("Bandas:", nombres_banda)

rgb  = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
ndvi = img[6] / 10000.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))
ax1.imshow(rgb); ax1.set_title("Color verdadero (geomediana, marzo 2018)"); ax1.axis("off")
im = ax2.imshow(ndvi, cmap="RdYlGn", vmin=0, vmax=0.9)
ax2.set_title("NDVI — vigor del cultivo"); ax2.axis("off")
plt.colorbar(im, ax=ax2, shrink=0.8); plt.tight_layout(); plt.show()

## De píxeles a parcelas: segmentación Shepherd

![segmentación](anim/es/05_segmentation.svg)


In [ ]:
# Paso 3 — Segmentar el tile en parcelas homogéneas (~30-60 s)
import shepherd_wasm, time

t0 = time.time()
resultado = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0,
    fixedKMeansInit=True)
seg = resultado.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcelas en {time.time()-t0:.1f} s")

from scipy import ndimage
bordes = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = rgb.copy(); vis[bordes] = [1, 1, 0]
plt.figure(figsize=(7.5, 7.5)); plt.imshow(vis)
plt.title(f"{n_seg} parcelas (amarillo = fronteras)"); plt.axis("off"); plt.show()

## Cada parcela se convierte en una fila de números

![variables](anim/es/06_features.svg)


In [ ]:
# Paso 4 — Variables por parcela: media y desviación de las 13 bandas
seg_plano = seg.ravel()
conteos = np.bincount(seg_plano, minlength=n_seg + 1).astype(float)
conteos[conteos == 0] = 1

variables = np.zeros((n_seg + 1, len(nombres_banda) * 2), dtype=np.float32)
for b in range(len(nombres_banda)):
    vals = img[b].ravel().astype(np.float64)
    s1 = np.bincount(seg_plano, weights=vals, minlength=n_seg + 1)
    s2 = np.bincount(seg_plano, weights=vals * vals, minlength=n_seg + 1)
    media = s1 / conteos
    var = np.maximum(s2 / conteos - media**2, 0)
    variables[:, 2*b], variables[:, 2*b+1] = media, np.sqrt(var)

nombres_variables = [f"{n}_{s}" for n in nombres_banda for s in ("media", "desv")]
print(f"Tabla de variables: {variables.shape[0]-1} parcelas x {variables.shape[1]} variables")

## Etiquetas de campo: la verdad en terreno (con filtro de pureza)

El raster de etiquetas viene de **1,645 puntos reales de campo** levantados
en el Valle del Yaqui: trigo, maíz, garbanzo y más.

![etiquetas y pureza](anim/es/07_labels_purity.svg)


In [ ]:
# Paso 5 — Asignar etiquetas a parcelas (mayoría + pureza)
with rasterio.open(LABELS) as src:
    lab = src.read(1)
nombres_clase = {int(k): v for k, v in json.load(open(NOMBRES)).items()}

etiqueta_parcela = np.zeros(n_seg + 1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    valores = lab[(seg == sid) & (lab > 0)]
    unicos = np.unique(valores)
    if len(unicos) == 1:                    # parcela pura -> sirve para entrenar
        etiqueta_parcela[sid] = unicos[0]

ids_entrena = np.flatnonzero(etiqueta_parcela)
print(f"Parcelas etiquetadas puras: {len(ids_entrena)} de {n_seg}")
for cid, cname in nombres_clase.items():
    print(f"  {cname:12s}: {(etiqueta_parcela[ids_entrena] == cid).sum():3d} parcelas")

## Entrenar el clasificador

![entrenamiento](anim/es/08_training.svg)


In [ ]:
# Paso 6 — Random Forest (en el navegador; el pipeline completo usa TPOT/AutoML)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = variables[ids_entrena]
y = etiqueta_parcela[ids_entrena]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=42, stratify=y)
modelo = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
modelo.fit(X_tr, y_tr)
presentes = sorted(np.unique(y_te))
print(classification_report(
    y_te, modelo.predict(X_te),
    labels=presentes, target_names=[nombres_clase[c] for c in presentes]))

## El mapa de cultivos

![mapa de cultivos](anim/es/09_crop_map.svg)


In [ ]:
# Paso 7 — Clasificar TODAS las parcelas y pintar el mapa
pred = np.zeros(n_seg + 1, dtype=int)
pred[1:] = modelo.predict(variables[1:])
mapa_cultivos = pred[seg]

paleta = {1: "#c2703d", 2: "#65a30d", 3: "#a8a29e",
          4: "#86efac", 5: "#7c3aed", 6: "#eab308"}
rgb_mapa = np.zeros((*mapa_cultivos.shape, 3))
for cid, hx in paleta.items():
    rgb_mapa[mapa_cultivos == cid] = [int(hx[i:i+2], 16)/255 for i in (1, 3, 5)]

import matplotlib.patches as mpatches
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 6))
ax1.imshow(rgb); ax1.set_title("Geomediana (color verdadero)"); ax1.axis("off")
ax2.imshow(rgb_mapa); ax2.set_title("Mapa de cultivos predicho"); ax2.axis("off")
ax2.legend(handles=[mpatches.Patch(color=paleta[c], label=nombres_clase[c])
                    for c in sorted(nombres_clase)],
           loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()
print("Felicidades — clasificaste cultivos reales en tu navegador. 🌾")

## De este taller a producción

![pipeline completo](anim/es/10_full_pipeline.svg)

Todo lo que acabas de hacer, **[geocrop_analysis_mx](https://github.com/abxda/geocrop_analysis_mx)**
lo hace a escala:

| Aquí (navegador) | Pipeline completo |
|---|---|
| 1 tile, 1 mes | regiones completas, muchos meses + radar Sentinel-1 |
| datos incluidos | descarga en vivo de catálogos STAC (NASA / Planetary Computer / Earth Search) — **u opcionalmente Google Earth Engine** |
| Random Forest | TPOT (AutoML) |
| media/desv por banda | estadísticas zonales completas + tus propios rasters (MDE, clima…) como variables extra |

Se instala con `pip install -r requirements.txt` (Windows, Linux, macOS —
sin conda, sin compiladores) y tiene tutorial paso a paso.
